In [1]:
import numpy as np
import pandas as pd
import os

# Path to your transposed CSV
csv_path = r"d:\Users\user\Downloads\gdc_files\final_expression_transposed.csv"

# Load data
df = pd.read_csv(csv_path)

# First column is sample name
samples = df.iloc[:, 0].values
# Remaining columns are gene expression values
data = df.iloc[:, 1:].values

print("Original data shape:", data.shape)  # Expect (1223, 16072)

# Determine padding to get a square matrix
num_features = data.shape[1]
side = int(np.ceil(np.sqrt(num_features)))
square_size = side ** 2
padding = square_size - num_features

print(f"Nearest square: {side}x{side} = {square_size} | Padding required: {padding}")

# Pad each sample vector with zeros
padded_data = np.pad(data, ((0, 0), (0, padding)), 'constant')

# Reshape into (samples, height, width, 1)
X = padded_data.reshape(data.shape[0], side, side, 1)

# Normalize between 0 and 1
X = X / np.max(X)

print("Final tensor shape:", X.shape)  # (1223, 127, 127, 1)

# Save as NumPy array
output_folder = r"d:\Users\user\Downloads\Values"
np.save(os.path.join(output_folder, "X_gene_images.npy"), X)

# Optionally save sample names (for later matching)
np.save(os.path.join(output_folder, "sample_names.npy"), samples)

print(f"✅ Saved reshaped CNN-ready data to: {output_folder}")


Original data shape: (1223, 16071)
Nearest square: 127x127 = 16129 | Padding required: 58
Final tensor shape: (1223, 127, 127, 1)
✅ Saved reshaped CNN-ready data to: d:\Users\user\Downloads\Values


In [7]:
import numpy as np

X = np.load(r"d:\Users\user\Downloads\Values\X_gene_images.npy")
samples = np.load(r"d:\Users\user\Downloads\Values\sample_names.npy", allow_pickle=True)

print("X shape:", X.shape)  # should be (1223, 127, 127, 1)
print("First sample ID:", samples[0])
print("First sample matrix (5x5 block):")
print(X[0, :5, :5, 0])


X shape: (1223, 127, 127, 1)
First sample ID: 0019c951-16c5-48d0-85c8-58d96b12d330
First sample matrix (5x5 block):
[[0.32321822 0.01951617 0.38746706 0.26167036 0.22353395]
 [0.24483308 0.2903167  0.15565191 0.19443029 0.18558029]
 [0.28051977 0.31724415 0.22156138 0.20424859 0.10869777]
 [0.23750122 0.0127304  0.28507316 0.25232736 0.06324753]
 [0.24515421 0.217073   0.29683094 0.26577571 0.24877469]]


In [15]:
import numpy as np
import pandas as pd

# Paths
metadata_path = r"d:\Users\user\Downloads\Values\metadata.csv"
samples_path = r"d:\Users\user\Downloads\Values\sample_names.npy"
output_path = r"d:\Users\user\Downloads\Values\samples_with_labels.npy"

# Load sample IDs
sample_ids = np.load(samples_path, allow_pickle=True)

# Load metadata
metadata = pd.read_csv(metadata_path)

# Create label mapping (Tumor=1, Solid=0)
metadata['label'] = metadata['sample_type'].apply(lambda x: 1 if 'Tumor' in x else 0)
file_to_label = dict(zip(metadata['file_id'], metadata['label']))

# Map labels to sample_ids
labels = np.array([file_to_label[sid] if sid in file_to_label else -1 for sid in sample_ids])

# Optional: check for missing sample_ids
missing_count = np.sum(labels == -1)
print(f"Number of sample_ids not found in metadata: {missing_count}")

# Create a structured array with sample ID and label
structured_array = np.array(list(zip(sample_ids, labels)), dtype=[('sample', 'U50'), ('label', 'i4')])

# Save the new structured array
np.save(output_path, structured_array)

print(f"Structured array saved with shape: {structured_array.shape}")


Number of sample_ids not found in metadata: 0
Structured array saved with shape: (1223,)


In [17]:
import numpy as np

# Load the labels array
labels = np.load(r"d:\Users\user\Downloads\Values\samples_with_labels.npy")

# Print the entire array
print(labels)

# Optional: print a summary if it's large
print("Shape:", labels.shape)
print("First 10 labels:", labels[:10])
print("Counts of each label:", np.unique(labels, return_counts=True))


[('0019c951-16c5-48d0-85c8-58d96b12d330', 1)
 ('0022cd20-f64f-4773-b9ff-a3de0b71b259', 1)
 ('00469928-b243-4cae-acd7-134508e99ceb', 1) ...
 ('ff570f9a-a252-496d-a452-344063851a7b', 1)
 ('ff5f8ada-17c5-497e-9182-63a05e3ab4c5', 1)
 ('ffeede9a-d9a9-4836-8c56-39f12a5fde0e', 1)]
Shape: (1223,)
First 10 labels: [('0019c951-16c5-48d0-85c8-58d96b12d330', 1)
 ('0022cd20-f64f-4773-b9ff-a3de0b71b259', 1)
 ('00469928-b243-4cae-acd7-134508e99ceb', 1)
 ('0081f507-b104-4214-9ea1-31dd69130991', 1)
 ('0094f9d0-45ec-4aad-bca0-71c60bdd7113', 1)
 ('00b13ccf-ad7c-4613-8366-7c583a399691', 1)
 ('010e405c-b91d-4046-898e-105d5830d9a9', 1)
 ('017d71aa-0999-4d8e-9cb4-88b9013e61eb', 1)
 ('02a0b95f-ea0f-4c0d-ba2d-9724ed9ad712', 1)
 ('02a11397-2a9b-4dba-8c6a-6b9f0aa24372', 1)]
Counts of each label: (array([('0019c951-16c5-48d0-85c8-58d96b12d330', 1),
       ('0022cd20-f64f-4773-b9ff-a3de0b71b259', 1),
       ('00469928-b243-4cae-acd7-134508e99ceb', 1), ...,
       ('ff570f9a-a252-496d-a452-344063851a7b', 1),
      

NameError: name 'y' is not defined